# LiFE Pulse (HC26-GEDA01) — ML Model Training Suite
### Mission LiFE Citizen Engagement App | Google Colab Master Training Notebook

This notebook trains and exports the core AI/ML pipelines for **LiFE Pulse**:
1. **M1:** Household Monthly kWh Estimator (Gradient Boosting / XGBoost)
2. **M4:** Proof Photo Classifier (MobileNetV3 Transfer Learning for LED bulbs, cloth bags, steel bottles, waste bins)
3. **M5:** Consumption Anomaly Detection (Rolling Z-score / Isolation Forest)
4. **M6:** Mission LiFE Multilingual RAG Vector Embeddings

In [ ]:
# Step 0: Environment Setup
!pip install -q scikit-learn numpy joblib torch torchvision sentence-transformers

## 1. M1: Household Monthly kWh Estimator
Calibrated physics + stochastic regression trained on 10,000 synthetic Indian urban households.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib, json

rng = np.random.default_rng(42)
N = 10000

members = rng.integers(1, 8, N)
home_type = rng.choice([0, 1, 2, 3], N, p=[.25, .4, .25, .1])
ac_count = rng.choice([0, 1, 2, 3], N, p=[.35, .35, .2, .1])
ac_star = rng.integers(1, 6, N)
ac_hours = rng.uniform(0, 10, N)
fridge = rng.choice([0, 1, 2], N, p=[.05, .85, .10])
geyser = rng.choice([0, 1], N, p=[.4, .6])
geyser_hours = rng.uniform(0.2, 2, N)
fans = rng.integers(1, 8, N)
led_share = rng.uniform(0, 1, N)
lights = rng.integers(4, 25, N)
washing = rng.choice([0, 1], N, p=[.3, .7])
ev = rng.choice([0, 1], N, p=[.93, .07])
season = rng.integers(0, 3, N)

kwh = (
    ac_count * ac_hours * 1.4 * 30 * (1.2 - ac_star * 0.05) * np.where(season == 1, 1.3, 0.6)
    + geyser * geyser_hours * 2.0 * 30 * np.where(season == 0, 1.2, 0.5)
    + fridge * 45.0
    + fans * 0.075 * 10 * 30
    + lights * (led_share * 9 + (1 - led_share) * 40) / 1000 * 5 * 30
    + washing * 8.0
    + ev * 90.0
    + members * 14.0
)
y = np.clip(kwh * rng.normal(1.0, 0.06, N), 35, 1400)
X = np.column_stack([
    members, home_type, ac_count, ac_star, ac_hours,
    fridge, geyser, geyser_hours, fans, led_share,
    lights, washing, ev, season
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = GradientBoostingRegressor(n_estimators=150, max_depth=5, learning_rate=0.08, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print('Test MAE:', mean_absolute_error(y_test, preds), 'kWh | R2 Score:', r2_score(y_test, preds))
joblib.dump(model, 'kwh_model.joblib')

## 2. M4: Proof Photo Classifier (Transfer Learning via MobileNetV3)
5 classes: `led_bulb`, `cloth_bag`, `steel_bottle`, `waste_bins`, `other`

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# MobileNetV3 Lightweight Classifier for Fast On-Device / CPU inference
classifier = models.mobilenet_v3_small(weights='DEFAULT')
num_classes = 5 # led_bulb, cloth_bag, steel_bottle, waste_bins, other
classifier.classifier[3] = nn.Linear(classifier.classifier[3].in_features, num_classes)
print('MobileNetV3 Proof Classifier ready for fine-tuning on verification photos.')

## 3. M5: Anomaly Detector (Rolling Z-Score & Isolation Forest)
Flags abnormal electricity spikes (>2.5 std over rolling 7-day baseline) for consumer leak/meter alerts.

In [ ]:
def detect_consumption_anomaly(daily_series, threshold=2.5):
    anomalies = []
    for i in range(7, len(daily_series)):
        window = daily_series[i-7:i]
        mean = np.mean(window)
        std = max(0.1, np.std(window))
        z = (daily_series[i] - mean) / std
        if z > threshold:
            anomalies.append((i, daily_series[i], z))
    return anomalies

sim_load = np.random.normal(8.0, 1.2, 30)
sim_load[15] = 22.5 # Injected spike (e.g. AC compressor failure or geyser left on)
alerts = detect_consumption_anomaly(sim_load)
print(f'Detected {len(alerts)} consumption anomalies:', alerts)